    # 🎤 VIDEO-FREE · 2. Que la canción la cante TU voz (RVC)
    [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/2_mi_voz_rvc.ipynb)

    Entrena un modelo con **tu propia voz** y lo usa para convertir la voz de la canción (cuaderno 1 o cualquier voz cantada) **a tu timbre**,
    respetando la melodía. Es la forma de que suene natural y que seas tú.

    - **Herramienta:** RVC, Retrieval-based Voice Conversion (licencia MIT; modelos base lj1995, MIT).
    - **Lo que necesitas:** **10 minutos o más de tu voz limpia**, sin música, sin eco y sin ruido. Mejor si es **cantada**. Pueden ser varios audios.
      Súbelos a Google Drive en la carpeta `VIDEO-FREE/mi_voz/`.
    - **GPU:** T4 sirve. El entrenamiento toma ~30–90 min según los minutos de voz y las épocas. El modelo se guarda en Drive para no repetirlo.
    - Usa solo tu propia voz o una voz con permiso de su dueño.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.

In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar RVC en un entorno aparte (10–15 min la primera vez)
import os
if not os.path.exists("/content/rvc"):
    !git clone -q https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI /content/rvc
    !git -C /content/rvc checkout -q 81eed5e8f68b6bed1789f682fe78cdd324495afc
%cd /content/rvc
# Quita los espejos chinos del archivo de requisitos y usa PyPI
!sed -i '/mirrors\./d' requirments_cu118_py312.txt
!pip install -q uv
!uv venv -q -p 3.12 /content/rvc_env
!uv pip install -q -p /content/rvc_env/bin/python "torch==2.7.1+cu118" "torchaudio==2.7.1+cu118" --index-url https://download.pytorch.org/whl/cu118
!uv pip install -q -p /content/rvc_env/bin/python -r requirments_cu118_py312.txt --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu118
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download("lj1995/VoiceConversionWebUI", allow_patterns=["hubert_base/*", "pretrained_v2/*"], local_dir="/content/rvc/assets")
hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", local_dir="/content/rvc/assets/rmvpe")
hf_hub_download("lj1995/VoiceConversionWebUI", "mute.zip", local_dir="/content/rvc/.model-downloads")
!cd /content/rvc && python -m zipfile -e .model-downloads/mute.zip logs
%cd /content
print("RVC listo")

In [ ]:
#@title 3 · Reunir tus grabaciones
#@markdown Toma los audios de `Drive/VIDEO-FREE/mi_voz/`. Si no tienes Drive, deja la casilla y súbelos al ejecutar.
DESDE_DRIVE = True  #@param {type:"boolean"}
import shutil, glob
from pathlib import Path
datos = Path("/content/rvc_datos"); datos.mkdir(exist_ok=True)
if DESDE_DRIVE and Path("/content/drive/MyDrive/VIDEO-FREE/mi_voz").exists():
    for f in glob.glob("/content/drive/MyDrive/VIDEO-FREE/mi_voz/*"):
        shutil.copy(f, datos)
else:
    from google.colab import files
    for nombre, contenido in files.upload().items():
        (datos / nombre).write_bytes(contenido)
total = sum(comun.duracion(f) for f in datos.iterdir())
print(f"{len(list(datos.iterdir()))} archivos, {total / 60:.1f} minutos de voz")
if total < 300:
    print("⚠️ Menos de 5 minutos: el resultado será pobre. Lo recomendado son 10 minutos o más.")

## 4 · Entrenar tu modelo de voz

Si ya entrenaste antes, **salta esta parte**: la celda 5 busca tu modelo en Drive.

Ejecuta la celda de abajo. Aparece un enlace `https://….gradio.live`: ábrelo y en la pestaña **Train** (Entrenar) pon:

| Campo | Valor |
|---|---|
| Experiment name | `mi_voz` |
| Target sample rate | `40k` |
| Pitch guidance | Sí |
| Version | `v2` |
| Training folder | `/content/rvc_datos` |
| Pitch extraction | `rmvpe` |
| Total epochs | 200 (100 si tienes poca voz) |

Pulsa en orden **Process data**, **Feature extraction**, **Train model** y **Train feature index**. Cuando termine, **detén esta celda** (botón ■) y sigue con la 5.

In [ ]:
#@title Abrir la interfaz de entrenamiento de RVC
!cd /content/rvc && /content/rvc_env/bin/python webui.py --colab --noautoopen

In [ ]:
#@title 5 · Guardar o recuperar tu modelo en Drive
import glob, shutil
from pathlib import Path
drive_rvc = Path("/content/drive/MyDrive/VIDEO-FREE/rvc"); drive_rvc.mkdir(parents=True, exist_ok=True)
pesos = sorted(glob.glob("/content/rvc/assets/weights/mi_voz*.pth"))
indices = sorted(glob.glob("/content/rvc/logs/mi_voz/added_*.index")) + sorted(glob.glob("/content/rvc/assets/indices/mi_voz*.index"))
if pesos:
    shutil.copy(pesos[-1], drive_rvc / "mi_voz.pth")
    if indices: shutil.copy(indices[-1], drive_rvc / "mi_voz.index")
    print("Modelo guardado en Drive:", drive_rvc)
elif (drive_rvc / "mi_voz.pth").exists():
    print("Uso el modelo que ya tenías en Drive.")
else:
    raise SystemExit("No encuentro tu modelo. Entrena primero (paso 4).")
MODELO = str(drive_rvc / "mi_voz.pth")
INDICE = str(drive_rvc / "mi_voz.index") if (drive_rvc / "mi_voz.index").exists() else ""

In [ ]:
#@title 6 · Convertir la voz de la canción a tu voz y mezclar
PROYECTO = "carretera-sin-ti.json"  #@param {type:"string"}
#@markdown Semitonos para acomodar la melodía a tu rango (0 = igual; −12 una octava abajo; +12 una arriba).
TONO = 0  #@param {type:"slider", min:-12, max:12, step:1}
PARECIDO = 0.75  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown `PARECIDO` más alto copia más tu timbre; más bajo deja la voz más limpia.
VOLUMEN_VOZ_DB = 0  #@param {type:"slider", min:-6, max:6, step:1}

p = comun.proyecto(PROYECTO)
nombre = comun.slug(p.get("titulo", "cancion"))
ia = (p.get("ia") or {}).get("cancion") or {}
def conseguir(drive_nombre, repo_camino):
    ruta = Path("/content/drive/MyDrive/VIDEO-FREE") / drive_nombre
    if ruta.exists(): return ruta
    if repo_camino: return comun.bajar(repo_camino)
    return None
voz_ia = conseguir(f"{nombre}_voz_ia.wav", ia.get("voz"))
instrumental = conseguir(f"{nombre}_instrumental_ia.wav", ia.get("instrumental"))
if voz_ia is None:  # sin cuaderno 1: separa la canción actual del proyecto
    voz_ia, instrumental = comun.separar_voz(comun.bajar(p["audio"]))
salida_voz = comun.TRABAJO / f"{nombre}_mi_voz_sola.wav"
indice = f'--index "{INDICE}"' if INDICE else ""
comun.sh(f'/content/rvc_env/bin/python infer/cli.py --model "{MODELO}" {indice} --input "{voz_ia}" --output "{salida_voz}" '
         f'--pitch {TONO} --f0-method rmvpe --index-rate {PARECIDO} --protect 0.33 --overwrite', cwd="/content/rvc")
print("Tu voz:"); comun.escuchar(salida_voz)
mezcla = comun.mezclar(salida_voz, instrumental, comun.TRABAJO / f"{nombre}_mi_voz.wav", VOLUMEN_VOZ_DB)
final = comun.a_mp3(mezcla, comun.TRABAJO / f"{nombre}_mi_voz.mp3")
print("Canción con tu voz:"); comun.escuchar(final)

In [ ]:
#@title 7 · Guardar en tu repo
comun.a_drive(final); comun.a_drive(salida_voz)
comun.guardar(f"medios/audio/{final.name}", final.read_bytes(), f"Canción con mi voz: {p.get('titulo')}")
p["audio_anterior"] = p.get("audio")
p["audio"] = f"medios/audio/{final.name}"
p["ia"] = {**p.get("ia", {}), "mi_voz": {"modelo": "RVC v2", "tono": TONO, "parecido": PARECIDO}}
comun.guardar_proyecto(PROYECTO, p, f"Proyecto: canción con mi voz para {p.get('titulo')}")
print("Listo. En el portal pulsa «Recargar proyecto».")